# Lesson 04 — Mini V-JEPA on Moving MNIST (the main event)

**Read first:** `docs/04_vjepa_paper_walkthrough.md` and `docs/05_collapse_ema_and_training.md`

Everything from lessons 01–03, plus time. New ingredients relative to I-JEPA:
1. **tubelet tokenization** (lesson 01),
2. **3D multi-block masking** — spatial blocks repeated across ALL frames,
3. the paper's **schedules** (lr warmup+cosine, wd 0.04→0.4, EMA 0.996→1.0, 1.25× stretch).

The full training run takes **~30–60 min on a T4**. Save the checkpoint at the end — lesson 05 evaluates it.

This notebook uses the library (`src/vjepa_mini/`) rather than inline definitions. You have already built every component by hand; now read each file as it appears — the code is heavily commented and maps 1:1 to the paper.

In [ ]:
#@title Setup — run this first { display-mode: "form" }
# Works both on Colab (clones the repo) and locally (repo already present).
import os, sys

if os.path.exists("../src/vjepa_mini"):          # running locally from notebooks/
    sys.path.insert(0, os.path.abspath("../src"))
elif os.path.exists("world-model-jepa/src"):      # already cloned on Colab
    sys.path.insert(0, os.path.abspath("world-model-jepa/src"))
else:                                             # fresh Colab runtime
    REPO_URL = "https://github.com/josephlionel95-moon/world-model-jepa.git"
    os.system(f"git clone {REPO_URL}")
    sys.path.insert(0, os.path.abspath("world-model-jepa/src"))

import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if not torch.cuda.is_available():
    print("NOTE: Runtime > Change runtime type > T4 GPU for the training notebooks.")


## Part 1 — Meet the data

Moving MNIST: digits bouncing billiard-style. The generator keeps every clip's **generative factors** (digit class, velocity, direction) as labels — our ground truth for probing later. Infinite data: no clip repeats, like V-JEPA's 2M videos and unlike a fixed dataset.

In [ ]:
import torch
from torch.utils.data import DataLoader
from vjepa_mini import VJEPAConfig, MovingMNIST, VJEPA, Trainer
from vjepa_mini.data.moving_mnist import collate_clips
from vjepa_mini.eval.visualize import show_clip
from vjepa_mini.utils import set_seed, count_params

set_seed(0)
cfg = VJEPAConfig()
print(f"clip: {cfg.num_frames}x{cfg.img_size}x{cfg.img_size} -> "
      f"{cfg.grid_t}x{cfg.grid_h}x{cfg.grid_w} = {cfg.num_tokens} tokens")

ds = MovingMNIST(num_frames=cfg.num_frames, img_size=cfg.img_size)
clip, labels = ds[0]
print("clip", clip.shape, "| digit", labels["digit"].item(),
      "| direction bin", labels["direction"].item(),
      "| speed", f"{labels['speed'].item():.2f} px/frame")
show_clip(clip, title=f"digit {labels['digit'].item()}, moving & bouncing")

## Part 2 — See the masking

The prediction task IS the masking. Below: the same clip under short-range (8 blocks x 15%) and long-range (2 blocks x 70%) masks. Dimmed = hidden from the x-encoder = what the predictor must infer. Notice **the mask is identical in every frame** — read `src/vjepa_mini/data/masking.py` now for why (information leakage through temporal redundancy).

In [ ]:
from vjepa_mini.data.masking import MultiBlockMaskGenerator

grid = (cfg.grid_t, cfg.grid_h, cfg.grid_w)
short = MultiBlockMaskGenerator(*grid, num_blocks=cfg.short_range_num_blocks,
                                spatial_scale=cfg.short_range_scale)
long_ = MultiBlockMaskGenerator(*grid, num_blocks=cfg.long_range_num_blocks,
                                spatial_scale=cfg.long_range_scale)

for name, gen in [("short-range (8 x 15%)", short), ("long-range (2 x 70%)", long_)]:
    ctx, tgt = gen()
    print(f"{name}: {len(tgt)}/{cfg.num_tokens} tokens masked "
          f"({100*len(tgt)/cfg.num_tokens:.0f}%)")
    show_clip(clip, title=f"{name} — dimmed area is hidden from the encoder",
              mask=tgt, grid_hw=grid, patch=cfg.patch_size, tubelet_t=cfg.tubelet_t)

**Socratic pause.** The digit spends most frames NOT under the mask, yet with ~90% masked the visible context is a few thin strips. To predict target features the model must work out: which digit is present (appearance), and where it will be in each temporal slice (dynamics — position, velocity, bounces). Write down one piece of information the encoder can safely *discard* under this objective that a pixel model would have to keep. (Hint: stroke-level pixel noise; exact anti-aliasing of edges.)

## Part 3 — Build the model, sanity-check the pieces

In [ ]:
model = VJEPA(cfg)
print(f"encoder   {count_params(model.encoder)/1e6:.2f}M params")
print(f"predictor {count_params(model.predictor)/1e6:.2f}M params")
print(f"target encoder: EMA copy, requires_grad="
      f"{any(p.requires_grad for p in model.target_encoder.parameters())}")

# one forward pass, untrained
ctx_idx, tgt_idx = short()
out = model(clip.unsqueeze(0), ctx_idx, tgt_idx)
print({k: round(v.item(), 4) for k, v in out.items()})
# Read src/vjepa_mini/models/vjepa.py NOW — the whole method is one page.

## Part 4 — Train

Watch three numbers (docs/05 §5.6):
- **loss**: falls fast then slowly; must NOT crash to ~0 (that's collapse, not victory)
- **target_std**: the vital sign; sliding to 0 = collapse
- **lr / momentum**: just the schedules doing their thing

~30 min for 6000 steps on a T4. Coffee, or read the paper's Section 4 while it runs.

In [ ]:
loader = DataLoader(ds, batch_size=cfg.batch_size, num_workers=2,
                    collate_fn=collate_clips, drop_last=True,
                    persistent_workers=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
trainer = Trainer(model, cfg, loader, device, ckpt_dir="./checkpoints")
history = trainer.train()          # full run; pass total_steps=500 for a smoke run

In [ ]:
from vjepa_mini.eval.visualize import plot_history
plot_history(history)

## Part 5 — Quick qualitative check before full evaluation

PCA the token features of a fresh clip to RGB. A trained encoder should show the digit as a coherent colored region **that tracks its motion** across temporal slices; an untrained encoder shows soup. (Full quantitative evaluation is lesson 05.)

In [ ]:
from vjepa_mini.eval.visualize import pca_feature_map, plot_pca_map
from vjepa_mini.models.vit import VideoViT

test_clip, test_labels = MovingMNIST(seed=123)[0]
show_clip(test_clip, title="test clip")

pca = pca_feature_map(model.target_encoder, test_clip, device, grid)
plot_pca_map(pca, "trained V-JEPA features (PCA->RGB), one image per temporal slice")

random_enc = VideoViT(dim=cfg.enc_dim, depth=cfg.enc_depth, num_heads=cfg.enc_heads).to(device)
pca_r = pca_feature_map(random_enc, test_clip, device, grid)
plot_pca_map(pca_r, "random-init encoder features (the baseline that keeps you honest)")

## Part 6 — Save the checkpoint (Colab storage is ephemeral!)

In [ ]:
# Option A: download
from google.colab import files                      # skip if running locally
files.download("./checkpoints/vjepa_final.pt")

# Option B: save to Google Drive (uncomment)
# from google.colab import drive
# drive.mount('/content/drive')
# !cp ./checkpoints/vjepa_final.pt /content/drive/MyDrive/

## Part 7 — The sabotage suite at video scale

Short runs (600 steps each, ~4 min): confirm the collapse story holds for the full method. Compare `target_std` trajectories against your healthy run.

In [ ]:
import dataclasses

def short_run(**overrides):
    set_seed(0)
    c = dataclasses.replace(cfg, total_steps=600, warmup_steps=60, **overrides)
    m = VJEPA(c)
    t = Trainer(m, c, loader, device, log_every=100)
    return t.train(), m

# A: EMA off (momentum pinned at 0)
hist_a, _ = short_run(ema_start=0.0, ema_end=0.0)

# B: your turn — edit models/vjepa.py (or subclass) to remove target LayerNorm,
#    then to remove stop-grad (delete torch.no_grad and set requires_grad=True;
#    give the optimizer those params). Watch what happens. Revert afterwards!

import matplotlib.pyplot as plt
plt.plot([h["step"] for h in history], [h["target_std"] for h in history], label="healthy")
plt.plot([h["step"] for h in hist_a], [h["target_std"] for h in hist_a], label="EMA off")
plt.axhline(0.05, color="r", ls="--"); plt.xlim(0, 1000)
plt.xlabel("step"); plt.ylabel("target_std"); plt.legend()
plt.title("collapse vital sign, video scale"); plt.show()

## Exercises

1. **L1 vs L2** (one line in `models/vjepa.py`): 2 seeds each at 2000 steps. Compare loss stability and (after lesson 05) probe accuracy.
2. **Masking matters** (paper Table 4 in miniature): train with (a) short-range only, (b) long-range only, (c) both. Hold steps fixed. Probe all three in lesson 05. Predict the ranking first.
3. **Break temporal repetition**: modify `MultiBlockMaskGenerator` to sample an independent spatial mask per temporal slice (same ratio). Predict, then measure. This is the "information leakage" claim, tested by you.
4. **Schedule stretch trick**: set `schedule_scale=1.0` vs `1.25`. Any difference at this scale? (A negative result here is informative — think about why the trick might only matter at 90k steps.)
5. **Challenge — two digits**: `MovingMNIST(num_digits=2)` with occlusions. Does probe accuracy drop more for digit identity or direction? What does that say about what gets entangled when objects overlap?

Next: `05_probing_and_analysis.ipynb` — the verdict.